# Week 54, Day 2: Longest increasing subsequence

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w054d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import bisect

cards = [3, 7, 2, 8, 5, 6, 1, 9, 4]
tops = []
for x in cards:
    i = bisect.bisect_left(tops, x)
    if i == len(tops):
        tops.append(x)
    else:
        tops[i] = x
    print(x, tops)
print(len(tops))

In [ ]:
def lis(a):
    n = len(a)
    f, prev = [1] * n, [None] * n
    for i in range(n):
        for j in range(i):
            if a[j] < a[i] and f[j] + 1 > f[i]:
                f[i], prev[i] = f[j] + 1, j
    if not a:
        return []
    i = max(range(n), key=lambda k: f[k])
    out = []
    while i is not None:
        out.append(a[i])
        i = prev[i]
    return out[::-1]

print(lis([3, 7, 2, 8, 5, 6, 1, 9, 4]), lis([5, 4, 3]))

In [ ]:
import bisect

def lis_fast(a):
    tops, tops_idx, parent = [], [], [None] * len(a)
    for i, x in enumerate(a):
        k = bisect.bisect_left(tops, x)
        parent[i] = tops_idx[k - 1] if k > 0 else None
        if k == len(tops):
            tops.append(x)
            tops_idx.append(i)
        else:
            tops[k], tops_idx[k] = x, i
    out, i = [], tops_idx[-1] if tops_idx else None
    while i is not None:
        out.append(a[i])
        i = parent[i]
    return out[::-1]

print(lis_fast([3, 7, 2, 8, 5, 6, 1, 9, 4]), lis_fast(list(range(10, 0, -1))))

> **Key idea.** The longest increasing subsequence has an O(n²) DP, f(i) = 1 + the best f(j) over earlier smaller elements, with parent links for the subsequence. The O(n log n) patience method keeps tops[k], the smallest ending of a run of length k + 1, always sorted: binary-search each element with bisect_left, append or replace. tops isn't the subsequence; store indices and parents to recover one. Use bisect_right for non-decreasing runs; sort nesting problems by one dimension (ties descending in the other) and run LIS on the other. The LCS of two permutations is an LIS of positions, and the fewest decreasing sequences covering a list equals the LIS length.

### Exercise: Rising runs, slowly

Write `lis(a)`: one longest strictly increasing subsequence, by the O(n²) DP with parent links (when several j give the same length, keep the **first**; end at the first position with the best length). Then `lis_ends(a)`: the list f, the length of the longest increasing subsequence ending at each position.

In [ ]:
def lis(a):
    ___

def lis_ends(a):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef best(a):\n    for r in range(len(a), 0, -1):\n        for c in itertools.combinations(a, r):\n            if all(x < y for x, y in zip(c, c[1:])):\n                return r\n    return 0\ndef test_slow():\n    \"the n squared DP\"\n    assert lis([3, 7, 2, 8, 5, 6, 1, 9, 4]) == [3, 7, 8, 9] and lis([]) == [] and lis([5, 5, 5]) == [5]\n    assert lis_ends([3, 7, 2, 8, 5, 6, 1, 9, 4]) == [1, 2, 1, 3, 2, 3, 1, 4, 2]\n    rng = random.Random(1)\n    for _ in range(150):\n        a = [rng.randint(0, 9) for _ in range(rng.randint(0, 9))]\n        s = lis(a)\n        it = iter(a)\n        assert len(s) == best(a) and all(x < y for x, y in zip(s, s[1:])) and all(x in it for x in s)\n        assert max(lis_ends(a), default=0) == best(a)")

### Exercise: Patience

Write `lis_length(a, strict=True)`: the length of the longest increasing subsequence in O(n log n), strictly increasing by default or non-decreasing with `strict=False`. Then `lis_fast(a, strict=True)`: one such subsequence, with indices and parent links. The checks run a list of 100,000 numbers.

In [ ]:
import bisect

def lis_length(a, strict=True):
    ___

def lis_fast(a, strict=True):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef slow(a, strict):\n    f = [1] * len(a)\n    for i in range(len(a)):\n        for j in range(i):\n            if (a[j] < a[i] if strict else a[j] <= a[i]):\n                f[i] = max(f[i], f[j] + 1)\n    return max(f, default=0)\ndef test_fast():\n    \"patience\"\n    assert lis_length([3, 7, 2, 8, 5, 6, 1, 9, 4]) == 4 and lis_length([]) == 0\n    assert lis_length([2, 2, 2]) == 1 and lis_length([2, 2, 2], strict=False) == 3\n    rng = random.Random(2)\n    for _ in range(200):\n        a = [rng.randint(0, 6) for _ in range(rng.randint(0, 12))]\n        for strict in (True, False):\n            s = lis_fast(a, strict)\n            it = iter(a)\n            assert len(s) == lis_length(a, strict) == slow(a, strict)\n            assert all((x < y if strict else x <= y) for x, y in zip(s, s[1:])) and all(x in it for x in s)\n    big = [rng.randint(0, 10 ** 9) for _ in range(100_000)]\n    t0 = time.perf_counter()\n    s = lis_fast(big)\n    assert len(s) == lis_length(big) and all(x < y for x, y in zip(s, s[1:]))\n    assert time.perf_counter() - t0 < 5")

### Exercise: How many longest?

Write `count_lis(a)`: how many strictly increasing subsequences have the greatest possible length (counting by positions, so equal values at different positions make different subsequences). Use the O(n²) DP with `length[i]` and `count[i]`.

In [ ]:
def count_lis(a):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef brute(a):\n    for r in range(len(a), 0, -1):\n        n = sum(1 for c in itertools.combinations(a, r) if all(x < y for x, y in zip(c, c[1:])))\n        if n:\n            return n\n    return 1 if not a else 0\ndef test_count():\n    \"counting\"\n    assert count_lis([1, 3, 5, 4, 7]) == 2 and count_lis([2, 2, 2, 2]) == 4 and count_lis([]) == 1\n    rng = random.Random(3)\n    for _ in range(200):\n        a = [rng.randint(0, 6) for _ in range(rng.randint(0, 10))]\n        assert count_lis(a) == brute(a), a")

### Exercise: Envelopes inside envelopes

An envelope `(w, h)` fits inside another if it is strictly narrower **and** strictly shorter. Write `most_nested(envelopes)`: the most envelopes in one nest, in O(n log n): sort by width ascending and, for equal widths, height **descending**, then take the LIS of the heights. Then `nest(envelopes)`: one such nest as a list, smallest first.

In [ ]:
import bisect

def most_nested(envelopes):
    ___

def nest(envelopes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef slow(envs):\n    envs = sorted(envs)\n    f = [1] * len(envs)\n    for i in range(len(envs)):\n        for j in range(i):\n            if envs[j][0] < envs[i][0] and envs[j][1] < envs[i][1]:\n                f[i] = max(f[i], f[j] + 1)\n    return max(f, default=0)\ndef test_envelopes():\n    \"nesting\"\n    assert most_nested([(5, 4), (6, 4), (6, 7), (2, 3)]) == 3 and most_nested([(1, 1), (1, 1)]) == 1\n    assert most_nested([]) == 0 and nest([]) == []\n    rng = random.Random(4)\n    for _ in range(150):\n        envs = [(rng.randint(1, 8), rng.randint(1, 8)) for _ in range(rng.randint(0, 10))]\n        n = nest(envs)\n        assert most_nested(envs) == slow(envs) == len(n)\n        assert all(a[0] < b[0] and a[1] < b[1] for a, b in zip(n, n[1:]))\n        left = list(envs)\n        for e in n:\n            left.remove(e)")

### Exercise: LCS of permutations

Two lists hold the same distinct items in different orders. Write `common_order(a, b)`: the length of their LCS in O(n log n), by mapping each item of b to its position in a and finding the LIS of those positions. The checks compare with the O(n²) table on small lists and run 50,000 items.

In [ ]:
import bisect

def common_order(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef table(a, b):\n    n, m = len(a), len(b)\n    prev = [0] * (m + 1)\n    for i in range(1, n + 1):\n        cur = [0] * (m + 1)\n        for j in range(1, m + 1):\n            cur[j] = prev[j - 1] + 1 if a[i - 1] == b[j - 1] else max(prev[j], cur[j - 1])\n        prev = cur\n    return prev[-1]\ndef test_perms():\n    \"reduction\"\n    assert common_order(list(\"abcde\"), list(\"aebdc\")) == 3 and common_order([], []) == 0\n    rng = random.Random(5)\n    for _ in range(100):\n        a = list(range(rng.randint(0, 10)))\n        rng.shuffle(a)\n        b = a[:]\n        rng.shuffle(b)\n        assert common_order(a, b) == table(a, b)\n    a = list(range(50_000))\n    b = a[:]\n    rng.shuffle(b)\n    t0 = time.perf_counter()\n    assert common_order(a, b) > 300\n    assert time.perf_counter() - t0 < 5")

### Bug hunt: Equal isn't increasing

For a **strictly** increasing subsequence, an equal value must replace a pile top, not extend it. This uses `bisect_right`, so equal values pile up and it prints `4` for `[2, 2, 2, 2]`. Use `bisect_left` so it prints `1`.

In [ ]:
import bisect

tops = []
for x in [2, 2, 2, 2]:
    k = bisect.bisect_right(tops, x)
    if k == len(tops):
        tops.append(x)
    else:
        tops[k] = x
print(len(tops))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"strict\"\n    expect_output(\"1\")")

### Bug hunt: The pile tops are not the answer

This prints the patience pile tops as if they were the subsequence, but `[1, 4, 6, 9]` isn't a subsequence of the input at all (1 comes after 6). Keep indices and parent links and print a real one: `[2, 5, 6, 9]`.

In [ ]:
import bisect

a = [3, 7, 2, 8, 5, 6, 1, 9, 4]
tops = []
for x in a:
    k = bisect.bisect_left(tops, x)
    if k == len(tops):
        tops.append(x)
    else:
        tops[k] = x
print(tops)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a real subsequence\"\n    expect_output(\"[2, 5, 6, 9]\")")

### Bug hunt: Same width, both nested

Sorting envelopes by `(width, height)` puts `(6, 4)` before `(6, 7)` and lets the LIS of heights use both, though equal widths can't nest. It prints `4`; sort equal widths by height **descending** so it prints `3`.

In [ ]:
import bisect

envelopes = [(5, 4), (6, 4), (6, 7), (2, 3), (6, 5)]
tops = []
for w, h in sorted(envelopes):
    k = bisect.bisect_left(tops, h)
    if k == len(tops):
        tops.append(h)
    else:
        tops[k] = h
print(len(tops))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"ties descending\"\n    expect_output(\"3\")")

### Puzzle: Patience length

Arrange the lines into the O(n log n) length of the longest strictly increasing subsequence. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the LIS length of [10, 9, 2, 5, 3, 7, 101, 18]
q = ___   # tops after patience on [4, 1, 3], as a list
r = ___   # the non-decreasing LIS length of [1, 1, 2, 2]
s = ___   # the fewest decreasing sequences covering [1, 2, 3, 4]
t = ___   # True or False: tops is always a subsequence of the input

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 4 and q == [1, 3] and r == 4, \"p, q and r\"\n    assert s == 4 and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: platforms for trains

Trains arrive at a terminus in a given order, each with a departure number (lower leaves sooner). Each platform is a dead-end siding: trains on it leave in reverse order of arrival, so a train may only join a platform whose last train leaves **later** than it does. Write:

- `fewest_platforms(departures)`: the fewest platforms needed. Each platform holds a decreasing run of departures, so this is the fewest decreasing sequences covering the list: the LIS length (departures are distinct).
- `assign(departures)`: a list giving the platform number (from 0) of each train, using the patience rule: put each train on the **leftmost** platform whose last train leaves later, else open a new one. Use bisect on the platforms' last departures.
- `departure_order(departures, platforms)`: given an assignment, the order trains actually leave: repeatedly, the train that leaves soonest among the trains at the front (end) of the platforms. Return the departures in leaving order, and raise `ValueError` if the assignment is impossible (a train behind one that leaves later).

The checks confirm that patience uses as few platforms as possible and that trains then leave in sorted order.

In [ ]:
import bisect, heapq

def fewest_platforms(departures):
    ___

def assign(departures):
    ___

def departure_order(departures, platforms):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef valid(deps, plats):\n    last = {}\n    for d, p in zip(deps, plats):\n        if p in last and last[p] < d:\n            return False\n        last[p] = d\n    return True\ndef brute(deps):\n    n = len(deps)\n    for k in range(1, n + 1):\n        for plats in itertools.product(range(k), repeat=n):\n            if valid(deps, plats):\n                return k\n    return 0\ndef test_platforms():\n    \"trains\"\n    deps = [3, 1, 4, 5, 2]\n    assert fewest_platforms(deps) == 3 and assign(deps) == [0, 0, 1, 2, 1]\n    assert departure_order(deps, assign(deps)) == [1, 2, 3, 4, 5]\n    assert fewest_platforms([]) == 0 and assign([]) == [] and departure_order([], []) == []\n    try:\n        departure_order([1, 2], [0, 0])\n        assert False\n    except ValueError:\n        pass\n    rng = random.Random(6)\n    for _ in range(60):\n        deps = rng.sample(range(1, 20), rng.randint(1, 7))\n        plats = assign(deps)\n        assert fewest_platforms(deps) == brute(deps) == len(set(plats))\n        assert valid(deps, plats) and departure_order(deps, plats) == sorted(deps)\n    big = rng.sample(range(10 ** 6), 50_000)\n    plats = assign(big)\n    assert max(plats) + 1 == fewest_platforms(big) and departure_order(big, plats) == sorted(big)")

## Warm-up

Deal the cards one at a time onto piles: each card goes on the **leftmost** pile whose top is at least as big, or starts a new pile on the right. `tops` holds the pile tops, and it's always sorted, so binary search finds the pile. The number of piles at the end is the length of the longest increasing subsequence.

## The O(n²) DP

A subsequence keeps elements in their original order, with gaps allowed. Use last week's "best ending here":

**f(i) = the length of the longest increasing subsequence ending exactly at position i**.

The element before `a[i]` in such a subsequence is some `a[j]` with j < i and `a[j] < a[i]`, so

f(i) = 1 + max(f(j) for j < i with a[j] < a[i]), or 1 if there's no such j.

The answer is the maximum of f(i). Keeping `prev[i]`, the j that gave the maximum, lets you walk back from the best ending to recover the subsequence. There are n states and up to n choices each: O(n²). That's fine for a few thousand elements, and the structure (each element looks back at all earlier, smaller ones) carries over to harder variants where the fast method doesn't.

## Patience

The fast method keeps a list `tops`, where

**tops[k] = the smallest possible last element of an increasing subsequence of length k + 1 seen so far**.

A small last element is better: it leaves more room for later elements to extend it. Two facts make it work:

- `tops` is always **strictly increasing** (a length-3 run ends with something bigger than some length-2 run's end).
- A new element x extends the longest run whose end is smaller than x. In a sorted list, binary search finds that place: `i = bisect_left(tops, x)`. If i is the end of the list, x makes a longer run than any before (append); otherwise x becomes a smaller ending for runs of length i + 1 (replace `tops[i]`).

Each element costs one binary search: O(n log n) overall. The length of `tops` is the answer, but **`tops` itself is not a subsequence** in general: in the warm-up it ends as `[1, 4, 6, 9]`, and 1 comes after 6 in the input.

The name comes from the card game: David Aldous and Persi Diaconis's 1999 survey "Longest increasing subsequences: from patience sorting to the Baik-Deift-Johansson theorem" tells the story, from Colin Mallows's analysis in the 1960s to deep results in probability.

## Getting the subsequence back

To recover a subsequence, store **indices** instead of values: `tops_idx[k]` is the index of the element ending the best run of length k + 1, and `parent[i]` is the index of the element before i in its run: whichever element ended the best run one shorter at the moment i was placed. At the end, walk back from `tops_idx[-1]` through `parent`. Binary search needs the values, so keep a parallel list of the values (or search with a `key`).

## Variations

- **Non-decreasing** (equal values allowed): use `bisect_right`, so an equal value extends the run instead of replacing its end.
- **Decreasing**: negate the values, or reverse the comparison.
- **Counting** how many longest increasing subsequences there are: the O(n²) DP with a second array, `count[i]`, the number of best runs ending at i (add the counts of every j that achieves the maximum).
- **Two dimensions**, nesting envelopes or boxes `(w, h)`: sort by width, then find an increasing subsequence of heights. Equal widths can't nest, so sort them by height **descending**: then two envelopes of the same width can never both be in an increasing run of heights.
- **Chains** of pairs `(a, b)` where each must start after the previous ends: sort by end and be greedy (Week 52), or sort by start and use LIS.

## Reductions

The **LCS of two permutations** (each a rearrangement of the same distinct items) reduces to an LIS. Replace each item of b with its position in a; a common subsequence is then exactly an increasing run of those positions. O(n log n) instead of O(n²), which is how some diff tools speed up the common case of mostly unique lines (the Hunt-Szymanski algorithm, 1977, uses a similar idea).

And a covering fact: the **fewest decreasing sequences** that together use every element equals the length of the longest increasing subsequence. The patience piles are such a covering (each pile, read top to bottom, is increasing; read in order of placement it's decreasing), and no covering can do better, because two elements of an increasing run can't share a decreasing sequence. This is a form of Dilworth's theorem (Robert Dilworth, 1950).

## Practice

## Quiz

## Daily challenge

## Before you go

Shuffle a handful of playing cards and play patience with them: deal each onto the leftmost pile whose top is higher, else start a new pile. Count the piles, then find an increasing run of that length among the cards. Tomorrow: DP on trees and on graphs without cycles.